# Social-Media Platform Corpora -> LOKLOK Ground Truth

Automatic discovery + ingestion of **all available social-media platform datasets** on Kaggle
(X/Twitter, Reddit, YouTube, Instagram, Facebook, TikTok, Discord, Telegram, LinkedIn, general),
coded through the thesis codebook into aggregated ground-truth anchors compatible with
`governance/corpus.py` (`CorpusSummary.to_dict()`), `governance/evidence.py` (`EvidenceEntry`),
the MC synthesis (`mc_ground_truth`) and the `persona_engine` corpus loaders.

## Honesty contract
1. **Aggregated statistics only.** Sampled rows are stored locally (`payload.ndjson`) for the coding
   pipeline, but the thesis only ever consumes the per-corpus **aggregate anchors** (prevalence of
   trust / social-validation / quality / misinfo / engagement language). Licenses are captured per
   dataset; `unknown` is reported, never assumed.
2. **No fabrication.** An anchor is only emitted if it was *actually coded* (real keyword hits) or
   derived from an observed label/rating column. A platform-use anchor (`smu=1.0`) is always flagged
   `semanticDiscount` and never asserted as measured behaviour.
3. **Deterministic.** All sampling/ranking uses a fixed seed; identical Kaggle API results reproduce
   identical digests.
4. **Reproducible.** The notebook writes its Kaggle API search results into `provenance.json` so the
   exact dataset refs, fetched-at timestamps and row counts are auditable.

## Output drop (written to `/kaggle/working/social_media_ground_truth/`)
| file | contents | consumed by |
|---|---|---|
| `corpora/<id>/payload.ndjson` | sampled coded rows | persona_engine `load_corpora` (generic ndjson) |
| `corpora/<id>/provenance.json` | Kaggle metadata, license, fetchedAt, counts, seed | audit |
| `corpora/<id>/summary.json` | `CorpusSummary.to_dict()` per corpus | corpus.py digest |
| `summaries.json` | all corpus summaries | corpus.py / MC synthesis |
| `evidence-entries.json` | `EvidenceEntry` dataclass dicts | governance/evidence.py `register_evidence` |
| `ground_truth_bundle.json` | summaries + evidence + provenance in one file | self-report analysis Colab |
| `catalog.md` / `README.md` | tables + integration guide | researcher |
| `social_media_ground_truth.zip` | everything above | download & drop into repo |


## How to run
1. Open this notebook on Kaggle and **Connect** a CPU/GPU session (search + parse only, no heavy AI).
2. Run **Runtime → Run all**. At the end, download `social_media_ground_truth.zip`:
   `File → Save a copy → Output` (or grab the `social_media_ground_truth/` folder).
3. Unzip the bundle into the thesis repo under `data/ground_truth/corpora/` (each `<id>/` folder
   drops straight in; `persona_engine` picks `.ndjson` files generically).
4. From the repo, run `python -m governance.main evidence register-bundle` **after** the integration
   hook in the README is added (see final README cell) **or** simply `summaries.json` is consumed by
   a new `corpus.py` summarizer registered for each id.

**Dataset search needs Kaggle credentials.** If discovery ends up with only the curated refs, add a
token: set `KAGGLE_USERNAME` / `KAGGLE_KEY` as Kaggle *Secrets* (or place `~/.kaggle/kaggle.json`),
then re-run. Downloads via `kagglehub` need no credentials.


## Anchor contract (same as `governance/qual_codebook.py` + `mc_ground_truth.py`)

One **global codebook** maps keywords -> (code, anchor, direction):

| anchor | meaning | positive codes | negative codes |
|---|---|---|---|
| `BR` | Broadcast Rigor / trust | host_trust | host_mistrust |
| `SV` | Social Validation | peer_validation | — |
| `IQ` | Information Quality | content_quality | content_quality_neg |
| `CRIT` | Criticality / misinfo awareness | misinfo_concern | — |
| `PB` | Ride / podcast experience | ride_positive | ride_negative |
| `ENG` | Engagement willingness | willingness | willingness_neg |
| `LIST` | Audio listening | audio_use | — |
| `MOB` | Mobility / commute | commute | — |
| `SMU` | Social-media use | sm_use | — |

Anchors are aggregated per corpus as in `qual_codebook.aggregate`: direction balance -> smoothed
`value` on [0,1], `hits` = pos+neg, `weight` = hits. The MC synthesis (and persona engine) treats
`semanticDiscount` as the honesty multiplier when an anchor is a proxy (e.g. platform-use assumed
from the corpus provenance rather than measured).

In [ ]:
# =============================================================
# 0. CONFIG / CONTRACT CONSTANTS
# =============================================================
import os, json, re, math, random, hashlib, zipfile, io
from collections import Counter, defaultdict
from pathlib import Path
from datetime import datetime, timezone

try:
    import numpy as np
except Exception:
    np = None

try:
    import pandas as pd
except Exception:
    pd = None

SEED = 20260912
random.seed(SEED)
if np is not None:
    np.random.seed(SEED)

# output roots
WROOT = Path("/kaggle/working") if os.path.exists("/kaggle/working") else Path(".")
OUT = WROOT / "social_media_ground_truth"
CORPORA_OUT = OUT / "corpora"
CORPORA_OUT.mkdir(parents=True, exist_ok=True)

# runtime budgets (raise MAX_DATASETS / MAX_ROWS to go larger)
MAX_DATASETS = 16          # corpora processed this run
MAX_TOTAL_ROWS = 250_000   # global sampled-row budget
MAX_ROWS_PER_DATASET = 30_000
MAX_BYTES_PER_FILE = 600_000_000   # skip huge files
SKIP_EXISTING = True       # don't re-download if payload.ndjson already present

# platform taxonomy: (tokens for search + inference)
PLATFORMS = {
    "x_twitter":     ["twitter", "tweets", "x twitter"],
    "reddit":        ["reddit"],
    "youtube":       ["youtube"],
    "instagram":     ["instagram"],
    "facebook":      ["facebook"],
    "tiktok":        ["tiktok"],
    "discord":       ["discord"],
    "telegram":      ["telegram"],
    "linkedin":      ["linkedin"],
    "general_social": ["social media", "social network", "hashtag", "viral posts"],
}

# intents appended to each platform query for discovery
INTENTS = ["sentiment", "comments", "posts", "engagement", "text", "dataset"]

# a few very-well-known refs (searched dynamically too; failures are skipped)
CURATED_REFS = [
    "kazanova/sentiment140",     # ~1.6M tweets with sentiment labels
    "datasnaek/reddit",          # top reddit posts/comments
]

TEXT_KW = ["text", "tweet", "comment", "review", "body", "content", "status",
           "message", "headline", "title", "caption", "description", "post",
           "clean_text", "sentence", "statement", "quote", "bio", "lyric",
           "subreddit_text", "tweets", "comments"]
LABEL_KW = ["sentiment", "label", "target", "score", "rating", "class",
            "emotion", "polarity", "stars", "recommend", "category"]
SKIP_COL = ["url", "link", "date", "time", "timestamp", "user", "followers",
            "likes", "retweet", "share", "count", "id", "location", "image",
            "photo", "video", "media"]
DATA_EXTS = {".csv", ".tsv", ".json", ".jsonl", ".ndjson", ".parquet", ".txt"}

NOW = datetime.now(timezone.utc).isoformat(timespec="seconds")
print("root:", OUT)
print("pandas:", bool(pd), "numpy:", bool(np))


In [ ]:
# =============================================================
# 1. CODEBOOK  (port of governance/qual_codebook.py: EN+AR lexicons)
# =============================================================
EN = {
    "peer_validation": ["friend", "friends", "share", "shares", "follow", "like", "ok?",
                        "social proof", "rating", "popular", "audience counter"],
    "host_trust": ["trust", "trustworthy", "credible", "verify", "verif", "fact-check",
                   "sources", "corrects", "correction", "honest", "reliable", "badge",
                   "transparen"],
    "host_mistrust": ["exaggerate", "clickbait", "fake", "not trust", "distrust",
                      "sensational", "not verify", "false claim"],
    "content_quality": ["informative", "useful", "valuable", "quality", "insightful",
                        "depth", "original", "well-informed", "educational"],
    "content_quality_neg": ["boring", "shallow", "repetitive", "low-quality", "clutter"],
    "misinfo_concern": ["misinformation", "propaganda", "false", "verify", "believe",
                        "cross-check", "misleading", "aware", "spread"],
    "ride_positive": ["ride", "comfortable", "safe", "on-time", "driver", "clean",
                      "convenience"],
    "ride_negative": ["delay", "unsafe", "cancel", "expensive", "rude", "traffic",
                      "waiting"],
    "willingness": ["would use", "interested", "excited", "join", "subscribe", "try",
                    "come back", "come back more", "keep the audio on"],
    "willingness_neg": ["would not", "skip", "delete", "leave", "not use"],
    "audio_use": ["radio", "podcast", "listen", "audio", "talk show", "music",
                  "episode", "station"],
    "commute": ["uber", "careem", "taxi", "commute", "drive", "driving", "ride"],
    "sm_use": ["instagram", "tiktok", "facebook", "whatsapp", "snapchat", "scroll",
               "feed", "reels", "stream", "social media", "youtube"],
}

AR = {
    "peer_validation": ["صديق", "أصدقاء", "مشاركة", "متابعة", "إعجاب", "تقييم",
                        "جمهور", "شعبية"],
    "host_trust": ["ثقة", "موثوق", "مصداقية", "تحقق", "مصادر", "تصحيح", "شفافية",
                   "صادق", "شارة"],
    "host_mistrust": ["مبالغة", "عنوان مضلل", "مزيف", "غير موثوق", "خبر كاذب"],
    "content_quality": ["مفيد", "قيم", "جودة", "عميق", "أصلي", "تعليمي", "غني"],
    "content_quality_neg": ["ممل", "سطحية", "مكرر", "فوضوي"],
    "misinfo_concern": ["أخبار كاذبة", "دعاية", "مضلل", "تحقق", "انتشار", "وعي"],
    "ride_positive": ["مريحة", "آمنة", "في الموعد", "سائق محترم", "نظيفة"],
    "ride_negative": ["تأخر", "غير آمن", "إلغاء", "غالٍ", "وقح", "زحمة"],
    "willingness": ["سأستخدم", "مهتم", "متحمس", "سأشترك", "سأجرب", "سأعود"],
    "willingness_neg": ["لن أستخدم", "سأغادر", "سأحذف", "غير مهتم"],
    "audio_use": ["راديو", "بودكاست", "موسيقى", "استماع", "برنامج حوارى", "محطة"],
    "commute": ["أوبر", "كريم", "تاكسي", "تنقل", "قيادة", "سيارة"],
    "sm_use": ["انستجرام", "تيك توك", "فيسبوك", "واتساب", "سكرول", "منشورات", "رييلز"],
}

CODEBOOK = [
    ("peer_validation", "SV", +1, "social proof / followers / shared recommendations"),
    ("host_trust", "BR", +1, "host/trust signal language"),
    ("host_mistrust", "BR", -1, "distrust / exaggeration / clickbait"),
    ("content_quality", "IQ", +1, "perceived content value / substance"),
    ("content_quality_neg", "IQ", -1, "perceived poor content"),
    ("misinfo_concern", "CRIT", +1, "misinformation awareness / verification"),
    ("ride_positive", "PB", +1, "positive ride experience"),
    ("ride_negative", "PB", -1, "negative ride experience"),
    ("willingness", "ENG", +1, "willingness to engage / subscribe"),
    ("willingness_neg", "ENG", -1, "refusal / skip / leave"),
    ("audio_use", "LIST", +1, "radio / podcast listening"),
    ("commute", "MOB", +1, "rides / commute / driving"),
    ("sm_use", "SMU", +1, "social media use"),
]

ANCHOR_NAMES = {
    "BR": "Broadcast Rigor", "SV": "Social Validation", "IQ": "Information Quality",
    "CRIT": "Criticality / misinformation awareness", "PB": "Ride (podcast) experience",
    "ENG": "Engagement willingness", "LIST": "Audio listening", "MOB": "Mobility",
    "SMU": "Social-media use",
}


def _tokens(text):
    words = {w.strip(".,!?;:'\"()[]{}") for w in re.split(r"[\s/]+", text.lower())}
    return {w for w in words if w}


def code_text(text):
    '''Return [(code, anchor, direction)] matched by EN+AR lexicons.

    English keywords match whole tokens; Arabic keywords use substring matching
    because of prefix morphology (ال- / بال- / لل- definite-article chains).
    '''
    tokens = _tokens(text)
    blob = text.lower()
    hits = []
    for code, anchor, direction, _desc in CODEBOOK:
        lex = set(EN.get(code, [])) | set(AR.get(code, []))
        multi = [k for k in lex if " " in k]
        single = lex - set(multi)
        ar = [k for k in single if any(ord(ch) > 127 for ch in k)]
        en = single - set(ar)
        if tokens & set(en) or any(k in blob for k in multi) \
                or any(k in blob for k in ar):
            hits.append((code, anchor, direction))
    return hits


def _smoothed(value, hits):
    '''Anchor proposal on [0,1] with shrinkage strength proportional to hits.'''
    k = 1 + hits
    return max(0.001, min(0.999, (0.5 * k + value * hits) / (2 * k)))


print("codebook codes:", len(CODEBOOK), "| en terms:", sum(len(v) for v in EN.values()),
      "| ar terms:", sum(len(v) for v in AR.values()))


In [ ]:
# =============================================================
# 2. DIGEST HELPERS  (CorpusSummary.to_dict + EvidenceEntry shapes)
# =============================================================

def anchor_proposal(name, value, hits, note, discount=1.0):
    v = max(0.0, min(1.0, float(value)))
    out = {"value": round(v, 4), "weight": int(hits), "hits": int(hits), "note": note}
    if discount < 1.0:
        out["semanticDiscount"] = discount
    return out


def dir_sha256(d: Path) -> str:
    parts = []
    for p in sorted(d.rglob("*")):
        if p.is_file():
            rel = str(p.relative_to(d)).replace("\\", "/")
            try:
                parts.append(f"{rel}:{hashlib.sha256(p.read_bytes()).hexdigest()}")
            except Exception:
                continue
    return hashlib.sha256("\n".join(parts).encode()).hexdigest() if parts else ""


def infer_platform(ref: str, title: str) -> str:
    blob = (ref + " " + title).lower()
    for plat in PLATFORMS:
        if any(tok in blob for tok in PLATFORMS[plat]):
            return plat
    return "general_social"


def slugify(name: str, maxlen: int = 40) -> str:
    s = re.sub(r"[^0-9a-z]+", "-", name.lower()).strip("-")
    return s[:maxlen].rstrip("-")


def write_ndjson(path: Path, rows: list):
    with open(path, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")


def write_json(path: Path, obj):
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, ensure_ascii=False)


In [ ]:
# =============================================================
# 3. KAGGLE ACCESS + DISCOVERY (searches "all available" datasets)
# =============================================================
api = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    print("kaggle API authenticated")
except Exception as e:
    print("kaggle API unavailable:", e)

try:
    import kagglehub
    print("kagglehub:", kagglehub.__version__ if hasattr(kagglehub, "__version__") else "ok")
except Exception:
    kagglehub = None


def _ds_meta(d):
    '''Defensively extract fields from a Kaggle Dataset object.'''
    get = lambda name, default=None: getattr(d, name, default)
    ref = get("ref")
    if ref is None:
        owner = get("ownerName") or get("owner_name") or ""
        slug = get("datasetSlug") or get("dataset_slug") or ""
        if owner and slug:
            ref = f"{owner}/{slug}"
    return {
        "ref": ref or "",
        "title": str(get("title") or ""),
        "subtitle": str(get("subtitle") or ""),
        "votes": int(get("totalVotes") or get("voteCount") or 0),
        "downloads": int(get("downloadCount") or 0),
        "bytes": int(get("totalBytes") or 0),
        "usability": get("usabilityRating") or "",
        "updated": str(get("lastUpdated") or ""),
    }


def discover(refs_seed=None):
    '''Union of curated refs + live Kaggle search per platform x intent.'''
    found = {}
    if refs_seed:
        for ref in refs_seed:
            if ref and "/" in ref:
                found[ref] = {"ref": ref, "title": ref, "subtitle": "curated",
                              "votes": 10**6, "downloads": 10**6, "bytes": 0,
                              "usability": "ok", "updated": "", "source": "curated"}
    if api is None:
        print("no Kaggle API -> using curated refs only")
        return list(found.values())

    seen_queries = set()
    for plat, tokens in PLATFORMS.items():
        for tok in tokens:
            for intent in INTENTS:
                q = f"{tok} {intent}".strip()
                if q in seen_queries:
                    continue
                seen_queries.add(q)
                try:
                    res = api.dataset_list(search=q, sort_by="votes")
                except Exception as e:
                    continue
                for d in res:
                    m = _ds_meta(d)
                    if not m["ref"]:
                        continue
                    m["source"] = f"search:{q}"
                    # keep best metadata per ref
                    prev = found.get(m["ref"])
                    if prev is None or m["votes"] > prev.get("votes", 0):
                        found[m["ref"]] = m
    print(f"discovered {len(found)} unique dataset refs across {len(PLATFORMS)} platforms")
    return list(found.values())


CANDIDATES = discover(refs_seed=CURATED_REFS)

# rank: vote/dataset quality first, then reasonable size, then usability
def _rank(c):
    usable = 0 if not c["usability"] else (2 if c["usability"] == "ok" else 1)
    size_ok = 0 if 0 < c["bytes"] > MAX_BYTES_PER_FILE else 1
    votes = min(c["votes"], 10**6)
    return (usable, votes, c["downloads"], size_ok, c["bytes"])


CANDIDATES.sort(key=_rank, reverse=True)
CANDIDATES = CANDIDATES[:MAX_DATASETS]

with open(OUT / "discovery.json", "w", encoding="utf-8") as fh:
    json.dump({"seed": SEED, "generatedAt": NOW, "candidates": CANDIDATES},
              fh, indent=2)
print("\nselected candidates (top", len(CANDIDATES), "):")
for c in CANDIDATES:
    print(" ", c["ref"], "| votes", c["votes"], "| MB",
          round(c["bytes"] / 1e6, 1), "|", c["usability"] or "-")


In [ ]:
# =============================================================
# 4. DOWNLOAD + TABLE INGESTION (csv/tsv/json/jsonl/ndjson/parquet/txt)
# =============================================================
def download_dir(ref: str) -> Path | None:
    '''Return a local dir with the downloaded dataset, or None.'''
    try:
        if kagglehub is not None:
            p = Path(kagglehub.dataset_download(ref))
            if p.exists():
                return p
        if api is not None:
            import tempfile
            tmp = Path(tempfile.mkdtemp(prefix="kgdl_"))
            api.dataset_download_files(ref, path=str(tmp), unzip=True)
            return tmp
    except Exception as e:
        print(f"  [download fail] {ref}: {e}")
    return None


def data_files(d: Path):
    out = []
    for p in sorted(d.rglob("*")):
        if not p.is_file():
            continue
        if p.suffix.lower() not in DATA_EXTS:
            continue
        if ".git/" in str(p) or p.name.lower().startswith(("readme", "license", "metadata")):
            continue
        sz = p.stat().st_size
        if 0 < sz <= MAX_BYTES_PER_FILE:
            out.append((sz, p))
    out.sort(reverse=True)   # largest data file first
    return [p for _sz, p in out]


def load_table(path: Path, sample_for_sniff=2000):
    '''Return a DataFrame for the best-understood file, else an empty list.'''
    if pd is None:
        return []
    try:
        suf = path.suffix.lower()
        if suf == ".parquet":
            df = pd.read_parquet(path)
        elif suf in (".json",):
            obj = json.loads(path.read_text(encoding="utf-8"))
            if isinstance(obj, dict):
                for key in ("data", "rows", "records", "items"):
                    if isinstance(obj.get(key), list):
                        obj = obj[key]; break
            if not isinstance(obj, list):
                return []
            df = pd.DataFrame(obj)
        elif suf in (".jsonl", ".ndjson"):
            rows = []
            with open(path, "r", encoding="utf-8") as fh:
                for line in fh:
                    line = line.strip()
                    if line:
                        try: rows.append(json.loads(line))
                        except Exception: pass
            df = pd.DataFrame(rows)
        elif suf in (".txt",):
            lines = [ln.strip() for ln in path.read_text(encoding="utf-8",
                                                         errors="replace").splitlines()
                     if ln.strip()]
            df = pd.DataFrame({"text": lines})
        else:  # csv / tsv
            sep = "\\t" if suf == ".tsv" else ","
            df = pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")
        df = df.dropna(axis=1, how="all")
        df = df.drop_duplicates().reset_index(drop=True)
        return df
    except Exception as e:
        print(f"  [parse fail] {path.name}: {e}")
        return []


In [ ]:
# =============================================================
# 5. COLUMN DETECTION: find the text column + optional sentiment label
# =============================================================
def detect_text_col(df):
    '''Return the best text column name, or None.'''
    scores = {}
    for col in df.columns:
        cl = str(col).lower()
        if any(k in cl for k in SKIP_COL):
            continue
        s = sum(1 for k in TEXT_KW if k in cl)
        if s == 0:
            continue
        scores[cl] = s
    if scores:
        best = max(scores, key=scores.get)
        # confirm it actually carries text
        vals = df[best].astype(str)
        if vals.str.len().mean() >= 3:
            return best
    # fallback: longest string column
    best_len, best_col = 0, None
    for col in df.columns:
        try:
            ml = df[col].astype(str).str.len().mean()
        except Exception:
            ml = 0
        if ml > best_len and ml > 20:
            best_len, best_col = ml, col
    return best_col


def detect_label_col(df, text_col):
    for col in df.columns:
        if col == text_col:
            continue
        cl = str(col).lower()
        if not any(k in cl for k in LABEL_KW):
            continue
        u = set(df[col].dropna().astype(str).str.lower().unique()[:40])
        if 1 < len(u) <= 12:
            return col
    return None


POS = {"positive", "pos", "good", "1", "4", "5", "1.0", "4.0", "5.0", "happy",
       "joy", "love", "yes", "support", "forward", "true", "1", "2", "3"}
NEG = {"negative", "neg", "bad", "0", "2", "1", "0.0", "2.0", "1.0", "sad",
       "anger", "hate", "no", "against", "false", "irrelevant"}


def label_shares(df, label_col):
    '''Return {'positiveShare','negativeShare','neutralShare','classes':{...}}.'''
    vals = df[label_col].dropna().astype(str).str.lower()
    counts = vals.value_counts().to_dict()
    total = max(1, len(vals))
    pos = sum(v for k, v in counts.items() if k in POS)
    neg = sum(v for k, v in counts.items() if k in NEG)
    neu = total - pos - neg
    return {
        "positiveShare": round(pos / total, 4),
        "negativeShare": round(neg / total, 4),
        "neutralShare": round(neu / total, 4),
        "classes": {k: int(v) for k, v in sorted(counts.items())},
    }


In [ ]:
# =============================================================
# 6. CODE A CORPUS + WRITE DIGEST
# =============================================================
global_sampled = 0

def process_corpus(cand, max_rows=MAX_ROWS_PER_DATASET):
    '''Download, sniff, sample deterministically, code, write digest.'''
    global global_sampled
    if pd is None:
        print("[skip] pandas required for ingestion")
        return None
    ref = cand["ref"]
    cid = slugify(ref.replace("/", "-"))
    cdir = CORPORA_OUT / cid
    payload_path = cdir / "payload.ndjson"

    if SKIP_EXISTING and payload_path.exists():
        print(f"[skip] {cid} already coded at {payload_path.name}")
        return cdir

    print(f"[process] {ref}")
    d = download_dir(ref)
    if d is None:
        return None
    files = data_files(d)
    if not files:
        print(f"  [no data] {ref}")
        return None

    df = None
    used_file = None
    for f in files:
        df = load_table(f)
        if df is not None and len(df) > 100:
            used_file = f
            break
        df = None
    if df is None or len(df) == 0:
        print(f"  [no rows] {ref}")
        return None

    n_total = len(df)
    text_col = detect_text_col(df)
    if text_col is None:
        print(f"  [no text column] {ref}")
        return None
    label_col = detect_label_col(df, text_col)

    # deterministic sample (stable across runs; bounded by global budget)
    sample_seed = (SEED + int(hashlib.sha256(ref.encode("utf-8"))
                              .hexdigest()[:8], 16)) % (2**32)
    budget = max(0, MAX_TOTAL_ROWS - global_sampled)
    df = df.sample(frac=1.0, random_state=sample_seed).reset_index(drop=True)
    df = df.iloc[:min(max_rows, budget, len(df))]
    n_sampled = len(df)

    text_series = df[text_col].astype(str).fillna("")

    coded_rows, hits = [], Counter()
    pos, neg = defaultdict(int), defaultdict(int)
    for i, t in enumerate(text_series):
        codes = code_text(t)
        for _code, anchor, direction in codes:
            hits[anchor] += 1
            (pos if direction > 0 else neg)[anchor] += 1
        coded_rows.append({
            "id": f"{cid}-{i}",
            "text": t,
            "platform": infer_platform(ref, cand["title"]),
            "codes": [c[0] for c in codes],
            "label": None if label_col is None else str(df.iloc[i][label_col]),
        })

    global_sampled += len(coded_rows)
    meta = {
        "ref": ref, "title": cand["title"], "subtitle": cand["subtitle"],
        "url": f"https://www.kaggle.com/datasets/{ref}" if ref else "",
        "license": getattr(d, "name", "") or cand.get("license") or "unknown",
        "platform": infer_platform(ref, cand["title"]),
        "seed": sample_seed,
        "generatedAt": NOW,
        "sourceDiscovery": cand.get("source", "curated"),
        "nTotalRows": int(n_total),
        "nSampledRows": int(n_sampled),
        "rowsCoded": int(len(coded_rows)),
        "textColumn": text_col,
        "labelColumn": label_col,
        "usedFile": str(used_file) if used_file else "",
        "codeHits": dict(hits),
    }

    # ---- aggregate anchors (same math as qual_codebook.aggregate) -------
    anchors = {}
    for anchor in sorted(set(pos) | set(neg)):
        p, n = pos.get(anchor, 0), neg.get(anchor, 0)
        balance = (p - n) / max(1, p + n)
        value = _smoothed(0.5 + 0.5 * balance, p + n)
        anchors[anchor] = anchor_proposal(
            anchor, value, p + n,
            f"coded text prevalence in {cid} ({ANCHOR_NAMES.get(anchor, anchor)})",
            discount=0.9)
    # platform-use proxy: the corpus is social-media text by construction
    anchors["SMU"] = anchor_proposal("SMU", 1.0, max(1, len(coded_rows)),
                                     f"platform-provenance proxy (Kaggle {cid})",
                                     discount=0.5)

    statistics = {
        "nRows": int(n_total),
        "nSampled": int(n_sampled),
        "textMeanLength": round(float(text_series.str.len().mean()), 1),
        "codedRowsShare": round(len(coded_rows) / max(1, n_sampled), 4),
        "codeHits": dict(hits),
    }
    if label_col:
        statistics["labelColumn"] = label_col
        statistics.update(label_shares(df, label_col))

    notes = [
        f"aggregated statistics only; sampled rows kept for coding reproducibility, "
        f"not redistributed beyond the thesis package",
        f"Kaggle dataset {ref} (license recorded in provenance.json)",
        f"coded through the thesis codebook ({len(CODEBOOK)} codes, EN+AR)",
        f"platform-use SMU anchor is a provenance proxy (semanticDiscount=0.5)",
    ]

    summary = {
        "id": cid, "title": f"Kaggle: {cand['title']} [{ref}]",
        "source_url": f"https://www.kaggle.com/datasets/{ref}" if ref else "",
        "license": meta["license"], "provenance": "secondary",
        "mode": "qualitative", "n": int(n_total),
        "anchors": anchors, "statistics": statistics, "notes": notes,
    }

    cdir.mkdir(parents=True, exist_ok=True)
    write_ndjson(payload_path, coded_rows)
    write_json(cdir / "provenance.json", meta)
    write_json(cdir / "summary.json", summary)
    print(f"  -> {cid}: n={n_total} sampled={n_sampled} text_col={text_col}"
          f" label={label_col} anchors={len(anchors)} hits={sum(hits.values())}")
    return cdir


RESULTS = {}
for cand in CANDIDATES:
    try:
        cdir = process_corpus(cand)
        if cdir is not None:
            cid = cdir.name
            with open(cdir / "summary.json", encoding="utf-8") as fh:
                RESULTS[cid] = json.load(fh)
    except Exception as e:
        print(f"[FAIL] {cand.get('ref')}: {e!r}")

print("\nprocessed", len(RESULTS), "corpora | total sampled rows:", global_sampled)


In [ ]:
# =============================================================
# 7. ASSEMBLE OUTPUTS (summaries.json / evidence-entries.json / bundle)
# =============================================================
SUMMARIES = RESULTS

evidence_entries = []
for cid, s in SUMMARIES.items():
    prov_path = CORPORA_OUT / cid / "provenance.json"
    meta = {}
    if prov_path.exists():
        with open(prov_path, encoding="utf-8") as fh:
            meta = json.load(fh)
    n_codes = int(s["statistics"].get("codeHits", {}).get("SMU", 0)) or int(s["n"])
    evidence_entries.append({
        "id": f"corpus-sm-{cid}",
        "name": s["title"],
        "kind": "open_corpus",
        "provenance": "secondary",
        "mode": "qualitative",
        "nature": "published",
        "n": int(s["n"]),
        "collected": meta.get("generatedAt", NOW)[:10],
        "location": f"Kaggle:{s['license']}",
        "instrument": "keyword-coded social-media corpus posts/comments (thesis codebook)",
        "artifact": f"data/ground_truth/corpora/{cid}/",
        "citation": s["source_url"],
        "weight": 0.45,
        "status": "ingested",
        "notes": "; ".join(s["notes"][:2]),
        "sha256": dir_sha256(CORPORA_OUT / cid),
    })

write_json(OUT / "summaries.json", {"seed": SEED, "generatedAt": NOW,
                                    "corpora": SUMMARIES})
write_json(OUT / "evidence-entries.json",
           {"seed": SEED, "generatedAt": NOW, "evidence": evidence_entries})

bundle = {
    "schemaVersion": "persona-engine-corpora-1",
    "generatedFor": "LOKLOK ground-truth synthesis + self-report Colab analysis",
    "seed": SEED, "generatedAt": NOW,
    "codebook": {code: {"anchor": a, "direction": d, "desc": desc}
                 for code, a, d, desc in CODEBOOK},
    "platforms": PLATFORMS,
    "corpora": SUMMARIES,
    "evidence": evidence_entries,
    "provenance": {cid: json.loads((CORPORA_OUT / cid / "provenance.json")
                                  .read_text(encoding="utf-8"))
                   for cid in SUMMARIES
                   if (CORPORA_OUT / cid / "provenance.json").exists()},
}
write_json(OUT / "ground_truth_bundle.json", bundle)

# ---- catalog table -----------------------------------------------------
lines = ["| corpus id | platform | n | anchors |", "|---|---|---|---|"]
for cid, s in sorted(SUMMARIES.items()):
    prov = (CORPORA_OUT / cid / "provenance.json")
    plat = "?"
    if prov.exists():
        with open(prov, encoding="utf-8") as fh:
            plat = json.load(fh).get("platform", "?")
    a = "; ".join(f"{k}={v['value']}" for k, v in s["anchors"].items())
    lines.append(f"| {cid} | {plat} | {s['n']} | {a} |")
with open(OUT / "catalog.md", "w", encoding="utf-8") as fh:
    fh.write("\n".join(lines) + "\n")
print("\n".join(lines))


In [ ]:
# =============================================================
# 8. VALIDATION, ZIP, README
# =============================================================
def validate(bundle):
    errors = []
    for cid, s in bundle["corpora"].items():
        if s["id"] != cid:
            errors.append(f"{cid}: id mismatch")
        for k, v in s["anchors"].items():
            if not (0.0 <= v["value"] <= 1.0):
                errors.append(f"{cid}:{k} value {v['value']} out of [0,1]")
            if not (v["weight"] >= 1):
                errors.append(f"{cid}:{k} weight<1")
        if s["n"] <= 0:
            errors.append(f"{cid}: non-positive n")
    return errors


errors = validate(bundle)
assert not errors, "\n".join(errors)

zip_path = OUT / "social_media_ground_truth.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.suffix == ".zip":
            continue
        z.write(p, arcname=p.relative_to(OUT))
print("zip:", zip_path, zip_path.stat().st_size / 1e6, "MB")

readme = '''# Social-Media Platform Corpora -> Ground Truth (Kaggle)
Generated {now} seed={seed}.

## What this is
Automatic discovery + ingestion of social-media platform datasets found on Kaggle,
coded through the thesis codebook into aggregated ground-truth anchors.

## Files
- summaries.json              all CorpusSummary.to_dict() digests (corpus.py compatible)
- evidence-entries.json       EvidenceEntry dicts (governance/evidence.py register_evidence)
- ground_truth_bundle.json    everything in one file (self-report Colab ready)
- corpora/<id>/payload.ndjson sampled rows (persona_engine generic corpus loader)
- corpora/<id>/provenance.json Kaggle metadata + license + seed
- corpora/<id>/summary.json   per-corpus digest
- discovery.json              raw Kaggle API search results
- catalog.md                  anchor table

## Integration (repo side)
1. Unzip into data/ground_truth/corpora/  (each <id>/ folder is self-contained).
2. persona_engine.load_corpora already ingests *.ndjson generically.
3. corpus.py: add each <id> to SUMMARIZERS via a loader that reads
   corpora/<id>/summary.json (a new `kaggle_corpora.py` adapter) and register
   each in SUMMARIZERS + CORPUS_SOURCE_ID.
4. governance/evidence.py: python -c "from governance.evidence import *; \\
   for e in json.load(open('data/ground_truth/corpora-kaggle/evidence-entries.json'))['evidence']: \\
   register_evidence(EvidenceEntry(**e))"
5. mc_ground_truth.py: add each id to DESIGN_QUALITY + collect_sources() via
   the kaggle adapter, then re-run `build_mc_version`.
6. Self-report Colab: read ground_truth_bundle.json for a full synthesis-ready view.
'''.format(now=NOW, seed=SEED)

with open(OUT / "README.md", "w", encoding="utf-8") as fh:
    fh.write(readme)

print("\n=== DELIVERABLES ===")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  {p.stat().st_size} bytes")

# quick integrity print
for cid, s in sorted(SUMMARIES.items()):
    anchors = "  ".join(f"{k}={v['value']:.3f}(w={v['weight']})"
                        for k, v in s["anchors"].items())
    print(f"\n{cid}  n={s['n']}\n  {anchors}")


## Done — integration next steps (see generated `README.md`)

1. **Download** `social_media_ground_truth.zip` from the notebook output and unzip into
   `data/ground_truth/corpora/` in the thesis repo.
2. `persona_engine` already ingests the `.ndjson` payloads generically (verified pattern:
   `_corpus_ndjson` for `*.ndjson`), so the **persona latent space** picks the new social-media
   text up the next time it runs.
3. For the **MC synthesis + evidence catalog**, the repo needs a tiny adapter
   (`governance/kaggle_corpora.py`) that adds each id to `SUMMARIZERS`, `CORPUS_SOURCE_ID`,
   `DESIGN_QUALITY` and `collect_sources()` reading `corpora/<id>/summary.json` — the digest
   shape matches `CorpusSummary.to_dict()` 1:1 (`anchors` keyed `BR/SV/IQ/CRIT/PB/ENG/LIST/MOB/SMU`,
   `provenance: secondary`, `mode: qualitative`, `n` = true row count).
4. **Self-report Colab** (you): consume `ground_truth_bundle.json` — it contains the same
   summaries + evidence entries + provenance in one documented bundle; return the analysis and
   I will integrate the anchors into `mc_ground_truth.collect_sources()` + the thesis tables.

> Honesty recap: anchors are **aggregated** coded prevalence; per-row text stays inside the
> corpus folder for reproducibility and is never redistributed; `SMU` platform proxy is always
> `semanticDiscount=0.5`; licenses as recorded by Kaggle (unknown where missing).